In [2]:
# STEP3_labeling.py

import pandas as pd
import numpy as np

def assign_label(row):
    """
    Label each row as BUY (0), HOLD (1), or SELL (2).

    Rules based on technical analysis expert logic:

    BUY  = Price is below MA20 AND RSI < 45 AND MACD > Signal
           (Stock is undervalued, momentum turning positive)

    SELL = Price is above MA20 AND RSI > 55 AND MACD < Signal
           (Stock is overvalued, momentum turning negative)

    HOLD = Everything else (unclear signal)
    """
    close     = row["Close"]
    ma5       = row["MA_5"]
    ma20      = row["MA_20"]
    rsi       = row["RSI"]
    macd      = row["MACD"]
    macd_sig  = row["MACD_Signal"]
    cci       = row["CCI"]
    stoch_k   = row["Stoch_K"]

    buy_signals = 0
    sell_signals = 0

    # Signal 1: Price vs MA
    if close < ma20 and ma5 < ma20:
        buy_signals += 1
    elif close > ma20 and ma5 > ma20:
        sell_signals += 1

    # Signal 2: RSI
    if rsi < 40:
        buy_signals += 1
    elif rsi > 60:
        sell_signals += 1

    # Signal 3: MACD crossover
    if macd > macd_sig:
        buy_signals += 1
    else:
        sell_signals += 1

    # Signal 4: CCI
    if cci < -80:
        buy_signals += 1
    elif cci > 80:
        sell_signals += 1

    # Signal 5: Stochastic
    if stoch_k < 25:
        buy_signals += 1
    elif stoch_k > 75:
        sell_signals += 1

    # Final decision: majority vote
    if buy_signals >= 3:
        return 0   # BUY
    elif sell_signals >= 3:
        return 2   # SELL
    else:
        return 1   # HOLD


df = pd.read_csv("indian_realty_features.csv", parse_dates=["Date"])

print("Assigning labels...")
df["Label"] = df.apply(assign_label, axis=1)

# Check label distribution
label_map = {0: "BUY", 1: "HOLD", 2: "SELL"}
print("\n📊 Label Distribution:")
counts = df["Label"].value_counts()
for k, v in counts.items():
    print(f"  {label_map[k]}: {v} samples ({v/len(df)*100:.1f}%)")

df.to_csv("indian_realty_labeled.csv", index=False)
print(f"\n✅ Labels assigned! Shape: {df.shape}")

Assigning labels...

📊 Label Distribution:
  HOLD: 11314 samples (48.2%)
  SELL: 6974 samples (29.7%)
  BUY: 5168 samples (22.0%)

✅ Labels assigned! Shape: (23456, 29)
